In [1]:
from tensorflow.keras.models import Model,load_model
from tensorflow.keras.applications import EfficientNetV2B2
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
from tensorflow.keras.layers import Dense,Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
from sklearn.metrics import classification_report,confusion_matrix
import clahe

I0000 00:00:1788274420.954587    1057 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788274421.959732    1057 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788274425.337054    1057 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
train_path = "../Datasets/forearm/Train"
test_path = "../Datasets/forearm/Test"

In [3]:
train_aug = ImageDataGenerator(
    preprocessing_function = lambda img : clahe.clahe_preprocess(img,preprocess_input),
    rotation_range = 10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.10,
    shear_range=0.05,
    validation_split = 0.2
)
valid_aug = ImageDataGenerator(
    preprocessing_function=lambda img: clahe.clahe_preprocess(img, preprocess_input),
    validation_split=0.2
)
test_aug = ImageDataGenerator(preprocessing_function = lambda img : clahe.clahe_preprocess(img,preprocess_input))

In [4]:
train_set = train_aug.flow_from_directory(
    train_path,
    target_size = (256,256),
    batch_size=8,
    class_mode="binary",
    subset="training",
    seed=42
)
validation_set = train_aug.flow_from_directory(
    train_path,
    target_size=(256, 256),
    batch_size=8,
    class_mode="binary",
    subset="validation",
    seed=42,
)
test_set = test_aug.flow_from_directory(
    test_path,
    target_size = (256,256),
    batch_size=8,
    class_mode="binary",
    shuffle = False
)

Found 1544 images belonging to 2 classes.
Found 386 images belonging to 2 classes.
Found 461 images belonging to 2 classes.


In [5]:
base_model = EfficientNetV2B2(include_top=False,weights="imagenet",input_shape=(256,256,3),pooling='avg')

I0000 00:00:1788274429.922861    1057 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1763 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6


In [6]:
base_model.trainable = False

In [7]:
x = base_model.output
x = Dense(64,activation='relu')(x)
output = Dense(1,activation='sigmoid')(x)

In [8]:
model = Model(inputs = base_model.input,outputs = output)

In [9]:
model.compile(optimizer= Adam(1e-3),loss="binary_crossentropy",metrics=['accuracy'])

In [10]:
erly_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

In [11]:
class_weight = {0:2,1:1.1}

In [12]:
model.fit(train_set,validation_data=validation_set,epochs=40,callbacks=erly_stop,class_weight=class_weight)

Epoch 1/40


I0000 00:00:1788274434.403816    1057 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1788274444.345260    3053 service.cc:153] XLA service 0x77fd60038eb0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1788274444.345349    3053 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 2050, Compute Capability 8.6 (Driver: 13.1.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1788274444.889019    3053 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1788274447.973221    3053 cuda_dnn.cc:461] Loaded cuDNN version 92400
I0000 00:00:1788274448.985671    3053 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_23525__.333
I0000 00:00:1788274474.826458    3053 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


193/193 ━━━━━━━━━━━━━━━━━━━━ 94s 278ms/step - accuracy: 0.6975 - loss: 0.8035 - val_accuracy: 0.7513 - val_loss: 0.4970
Epoch 2/40
193/193 ━━━━━━━━━━━━━━━━━━━━ 29s 151ms/step - accuracy: 0.7727 - loss: 0.6779 - val_accuracy: 0.7021 - val_loss: 0.5599
Epoch 3/40
193/193 ━━━━━━━━━━━━━━━━━━━━ 32s 166ms/step - accuracy: 0.7837 - loss: 0.6587 - val_accuracy: 0.7565 - val_loss: 0.4574
Epoch 4/40
193/193 ━━━━━━━━━━━━━━━━━━━━ 32s 168ms/step - accuracy: 0.7966 - loss: 0.5976 - val_accuracy: 0.7461 - val_loss: 0.4861
Epoch 5/40
193/193 ━━━━━━━━━━━━━━━━━━━━ 30s 157ms/step - accuracy: 0.7986 - loss: 0.5988 - val_accuracy: 0.7979 - val_loss: 0.4247
Epoch 6/40
193/193 ━━━━━━━━━━━━━━━━━━━━ 32s 165ms/step - accuracy: 0.8038 - loss: 0.5832 - val_accuracy: 0.7824 - val_loss: 0.4499
Epoch 7/40
193/193 ━━━━━━━━━━━━━━━━━━━━ 31s 161ms/step - accuracy: 0.8089 - loss: 0.5558 - val_accuracy: 0.8083 - val_loss: 0.3996
Epoch 8/40
193/193 ━━━━━━━━━━━━━━━━━━━━ 31s 161ms/step - accuracy: 0.8297 - loss: 0.5315 - val

In [13]:
prob = model.predict(test_set)

58/58 ━━━━━━━━━━━━━━━━━━━━ 29s 353ms/step


In [14]:
pred = (prob > 0.5).astype(int)

In [15]:
true_data = test_set.classes
model1 = model

In [16]:
print(classification_report(true_data,pred))
print(confusion_matrix(true_data,pred))

              precision    recall  f1-score   support

           0       0.90      0.74      0.81       235
           1       0.77      0.91      0.84       226

    accuracy                           0.83       461
   macro avg       0.84      0.83      0.83       461
weighted avg       0.84      0.83      0.83       461

[[175  60]
 [ 20 206]]


In [17]:
base_model.trainable = True

In [18]:
for layer in base_model.layers[:-30]:
    layer.trainable = False

In [19]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2
)

In [20]:
model.compile(optimizer=Adam(1e-5),loss="binary_crossentropy",metrics=['accuracy'])

In [21]:
model.fit(train_set,validation_data=validation_set,epochs=30,callbacks=[reduce_lr,early_stop],class_weight=class_weight)

Epoch 1/30


I0000 00:00:1788275101.732619    3053 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_86129__.357


193/193 ━━━━━━━━━━━━━━━━━━━━ 82s 234ms/step - accuracy: 0.8271 - loss: 0.5304 - val_accuracy: 0.7953 - val_loss: 0.4035 - learning_rate: 1.0000e-05
Epoch 2/30
193/193 ━━━━━━━━━━━━━━━━━━━━ 32s 162ms/step - accuracy: 0.8310 - loss: 0.5181 - val_accuracy: 0.8446 - val_loss: 0.3669 - learning_rate: 1.0000e-05
Epoch 3/30
193/193 ━━━━━━━━━━━━━━━━━━━━ 31s 159ms/step - accuracy: 0.8387 - loss: 0.5091 - val_accuracy: 0.8368 - val_loss: 0.3772 - learning_rate: 1.0000e-05
Epoch 4/30
193/193 ━━━━━━━━━━━━━━━━━━━━ 31s 159ms/step - accuracy: 0.8478 - loss: 0.4942 - val_accuracy: 0.8083 - val_loss: 0.4159 - learning_rate: 1.0000e-05
Epoch 5/30
193/193 ━━━━━━━━━━━━━━━━━━━━ 31s 161ms/step - accuracy: 0.8433 - loss: 0.4847 - val_accuracy: 0.8342 - val_loss: 0.4154 - learning_rate: 2.0000e-06
Epoch 6/30
193/193 ━━━━━━━━━━━━━━━━━━━━ 35s 182ms/step - accuracy: 0.8627 - loss: 0.4529 - val_accuracy: 0.8264 - val_loss: 0.3947 - learning_rate: 2.0000e-06
Epoch 7/30
193/193 ━━━━━━━━━━━━━━━━━━━━ 42s 215ms/step - 

In [22]:
prob = model.predict(test_set)

pred = (prob >= 0.5).astype(int)
true_data = test_set.classes
model1 = model

print(classification_report(true_data,pred))
print(confusion_matrix(true_data,pred))

58/58 ━━━━━━━━━━━━━━━━━━━━ 24s 274ms/step
              precision    recall  f1-score   support

           0       0.81      0.83      0.82       235
           1       0.81      0.80      0.81       226

    accuracy                           0.81       461
   macro avg       0.81      0.81      0.81       461
weighted avg       0.81      0.81      0.81       461

[[194  41]
 [ 46 180]]


In [23]:
# model.save('foreaam_x_ray_model.keras')